In [8]:
import Kea
from Kea.utils import plotting
from Kea.simulator import fbm
from Kea.statistics import statistics_base, moments
from Kea.statistics.spectra import strfn_spectra, spectra_base, arevalo_spectra, per_spectra
from Kea.statistics.statfunc import strfn, statfunc_base, statfunc_mpi_naive
from Kea.fitting import fit_base

import matplotlib as mpl
plotting.modify_rc()
import matplotlib.pyplot as plt

import numpy as np

D = 2
N = 256
#L = 1. ## STILL BAD...
#L = N
L = 5.*np.pi
grid_dims = [N for _ in range(D)]
phys_dims = [L for _ in range(D)]

twopi = 2.*np.pi
dk = twopi/L

In [9]:
min, max = -(D+3.), -1.
NUM = 20
alphas = np.linspace(max, min, NUM)
np.random.seed(1234)
fields = []
for a in alphas:
    fields.append(fbm.create_fbm(grid_dims, phys_dims, (a,), (1*dk,)))
energies = []
for f in fields:
    energies.append(moments.var(f, lenn=phys_dims))
specs = []
for i, f in enumerate(fields):
    kD, fekD = per_spectra.modal_spectrum(f, lenn=phys_dims)
    # Remove the Parseval's scaling factor I automatically apply
    fekD = fekD * (2.*np.pi/dk)**D
    specs.append((kD, fekD))
    #print(np.sum(fekD) * (dk/twopi)**D / energies[i])
    assert np.isclose(np.sum(fekD) * (dk/twopi)**D / energies[i], 1., rtol=0.001)

/home/m/Documents/science_codes/Kea/Kea/utils/funcs.py:73: RuntimeWarning: divide by zero encountered in reciprocal
  terms.append((xx/bb[0])**(-a[0]))
/home/m/Documents/science_codes/Kea/Kea/simulator/fbm.py:62: RuntimeWarning: invalid value encountered in multiply
  output = np.sqrt(output * C)
/home/m/Documents/science_codes/Kea/Kea/utils/funcs.py:73: RuntimeWarning: divide by zero encountered in power
  terms.append((xx/bb[0])**(-a[0]))


In [10]:
from scipy.stats import binned_statistic

def get_bins(min_bin, max_bin, nbins, max_half_bin_width=None, log_space=False):
    if log_space:
        bin_range = np.exp(np.linspace(np.log(min_bin), np.log(max_bin), nbins+1))
    else:
        bin_range = np.linspace(min_bin, max_bin, nbins+1)

    ## NOTE: Inherited code converts to integers; there are probably problems
    ##        with doing this... and also not doing this...
    ## valid_bins = np.unique(bin_range.astype(int))
    valid_bins = np.unique(bin_range.round(decimals=4))

    lower_bins, upper_bins = valid_bins[:-1], valid_bins[1:]

    bin_width = (upper_bins - lower_bins) / 2.
    if max_half_bin_width is not None:
        bin_width = np.minimum(bin_width, max_half_bin_width)

    bins = upper_bins - bin_width
    # consequences of not using the above note
    bin_edges = np.unique(np.concatenate((bins-bin_width, bins+bin_width)).round(decimals=4))
    return bins, bin_width, bin_edges

def wrap_mean(D, L, N):
    def mean(x):
        return np.nansum(x) * (L/np.sum(np.isfinite(x)))**D
    return np.nanmean

def wrap_sum(D, db):
    def sum(x):
        return np.nansum(x)# * (L/N)**(D-1) #/ (L/np.sum(np.isfinite(x)))**D
    return sum

def bin_data(nar, ar, mean_func='mean', std_func=np.nanstd,
             cut_excess=False, nan_small=False, min_bin=None,
             max_bin=None, bin_center=True, norm_bin_size=False, log_space=False,
             num_bins=None, ignore_nan=False, max_half_bin_width=None):
    # Find the basis of the position array
    pos = np.where(nar == 0)
    basis_index = [pos[i][0] for i in range(len(pos))]
    basis_index[0] = Ellipsis
    narbasis = nar[tuple(basis_index)]
    nn = narbasis[pos[0][0]:]

    # Calculate the bin space
    # Assume that the array is evenly space (this is an assumption made with everything)
    if min_bin is None:
        # If no minimum bin specified, then automatically choose one
        min_bin = np.nanmin(nn)

    if max_bin is None:
        max_bin = np.nanmax(nar)
    if isinstance(max_bin, str) and max_bin == 'basis':
        max_bin = np.nanmax(narbasis)

    if num_bins is None:
        binsize = np.diff(nn)[0]
        nbins = int(np.round(max_bin/binsize) + 1)
    else:
        nbins = num_bins

    bins, bin_widths, be = get_bins(min_bin, max_bin, nbins, max_half_bin_width, log_space)

    # Set bad, the regions outside the binning domain because scipy includes them into the binnings
    car = ar.copy()
    car[nar < min_bin] = np.nan
    car[nar > max_bin] = np.nan

    # Compute the binnings
    ar1d, bin_edges, _ = binned_statistic(nar.ravel(), car.ravel(), bins=be, statistic=mean_func)
    std1d, _, _ = binned_statistic(nar.ravel(), car.ravel(), bins=be, statistic=std_func)

    if nan_small:
        # Compute the counts, so we can ignore bad statistics
        cts, _, _ = binned_statistic(nar.ravel(), car.ravel(), bins=be, statistic='count')
        mask = cts <= 1
        ar1d[mask] = np.nan
        std1d[mask] = np.nan

    if bin_center:
       # Set the bins to the mid point of the bin edges
       bins1d = (bin_edges[1:] + bin_edges[:-1])/2.
    else:
       # Set the bins to the start of the bin edges
       bins1d = bin_edges[:-1]

    if max_half_bin_width is not None:
        # We should remove the bad bin edges (for the bins we don't want)
        mask = np.isin(bins1d, bins)
        ar1d = ar1d[mask]
        std1d = std1d[mask]
        bins1d = bins1d[mask]

    if norm_bin_size:
        # Divide the functions by their bin sizes
        # NOTE: The bin widths are actually half the bin widths
        if max_half_bin_width:
            # We have already calculate this
            width = 2.*bin_widths
        else:
            # Otherwise, calculate from binned_statistic bin_edges
            lower_bins, upper_bins = bin_edges[:-1], bin_edges[1:]
            width = (upper_bins - lower_bins)
            if max_half_bin_width is not None:
                width = np.minimum(width, 2.*max_half_bin_width)
        ar1d = ar1d / width
        std1d = std1d / width

    if cut_excess:
        # Cut off lags above the basis directions
        ar1d = ar1d[bins1d <= nn[-1]]
        std1d = std1d[bins1d <= nn[-1]]
        bins1d = bins1d[bins1d <= nn[-1]]

    if ignore_nan:
        # Remove nan values
        mask = np.isfinite(ar1d)
        ar1d = ar1d[mask]
        std1d = std1d[mask]
        bins1d = bins1d[mask]

    return bins1d, ar1d, std1d


In [11]:
def get_width(nar, nbins, log_space):
    pos = np.where(nar == 0)
    basis_index = [pos[i][0] for i in range(len(pos))]
    basis_index[0] = Ellipsis
    narbasis = nar[tuple(basis_index)]
    nn = narbasis[pos[0][0]:]

    # Calculate the bin space
    # Assume that the array is evenly space (this is an assumption made with everything)
    min_bin = twopi/L
    if min_bin is None:
        # If no minimum bin specified, then automatically choose one
        min_bin = np.nanmin(nn)

    max_bin = None
    if max_bin is None:
        max_bin = np.nanmax(nar)
    if isinstance(max_bin, str) and max_bin == 'basis':
        max_bin = np.nanmax(narbasis)

    if nbins is None:
        binsize = np.diff(nn)[0]
        nbins = int(np.round(max_bin/binsize) + 1)
    else:
        nbins = nbins

    _, width, _ = get_bins(min_bin, max_bin, nbins, log_space=log_space)
    return 2.*width

In [12]:
for e, kfek in enumerate(specs):
    kD, fekD = kfek
    ko, feko, _ = bin_data(spectra_base.wavenumber_mesh(kD), fekD,
        mean_func=np.nansum, min_bin=0., max_bin=None,
        ignore_nan=False, norm_bin_size=True, bin_center=True,
        cut_excess=False, nan_small=False)
    width = get_width(spectra_base.wavenumber_mesh(kD), None, False)
    e_estimation = (1./twopi)**D * np.nansum(feko * width)
    print(e_estimation / energies[e])
    #assert np.isclose(e_estimation / energies[e], 1., rtol=0.02), e_estimation / energies[e]

    for i in range(4, 8):
        ko, feko, _ = bin_data(spectra_base.wavenumber_mesh(kD), fekD,
            mean_func=np.nansum, min_bin=0., max_bin=None,
            ignore_nan=False, norm_bin_size=True, bin_center=True,
            cut_excess=False, nan_small=False, num_bins=N//i)
        width = get_width(spectra_base.wavenumber_mesh(kD), N//i, False)
        e_estimation = (1./twopi)**D * np.nansum(feko * width)
        print(e_estimation / energies[e])
        #assert np.isclose(e_estimation / energies[e], 1., rtol=0.02), e_estimation / energies[e]

    for i in range(4, 8):
        ko, feko, _ = bin_data(spectra_base.wavenumber_mesh(kD), fekD,
            mean_func=np.nansum, min_bin=2.*np.pi/L, max_bin=None,
            ignore_nan=False, norm_bin_size=True, bin_center=True,
            cut_excess=False, nan_small=False, num_bins=N//i, log_space=True)
        width = get_width(spectra_base.wavenumber_mesh(kD), N//i, True)
        e_estimation = (1./twopi)**D * np.nansum(feko * width)
        print(e_estimation / energies[e])
        #assert np.isclose(e_estimation / energies[e], 1., rtol=0.02), e_estimation / energies[e]


6.215394344632407
6.215393148432514
6.215397532149695
6.215395923053994
6.215395668712336
6.249922917675636
6.2499229176756375
6.249922917675637


/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


6.249922917675636
6.21543490343001
6.215436549378533
6.215441399870459
6.215438014691403
6.215436116078608
6.249964382930671
6.249964382930671


/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


6.249964382930671
6.249964382930669
6.2154202388090765
6.2154284461953875
6.2154338135725204
6.215435612638758
6.215434220684554
6.249964109941695


...

/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


6.2499999999408935
6.2499999999408935
6.215443246256958
6.21520569285798
6.215227934972962
6.215558424794146
6.215506603269285
6.249999999976833


/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


6.249999999976833
6.249999999976834
6.249999999976834
6.215443238472225
6.215205776258184
6.2152278418598454
6.215558561143467
6.215506706688236
6.249999999993277
6.24999999999328
6.249999999993278
6.249999999993278


/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/m/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1872: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [13]:
for e, kfek in enumerate(specs):
    kD, fekD = kfek
    km, fekm, _ = bin_data(spectra_base.wavenumber_mesh(kD), fekD,
                        mean_func=np.nanmean, min_bin=0., max_bin=None,
                        ignore_nan=False, norm_bin_size=False, bin_center=True,
                        cut_excess=False, nan_small=False)
    width = get_width(spectra_base.wavenumber_mesh(kD), None, False)

    if D == 1:
        dOmega = 2.
    if D == 2:
        dOmega = 2. * np.pi * (km)
    if D == 3:
        dOmega = 4. * np.pi * (km)**2

    e_estimation = np.nansum(fekm * dOmega * width) * (1./twopi)**D
    print(e_estimation / energies[e])
    #plt.loglog(ko, feko, marker='.')
    #assert np.isclose(np.nansum(fekm * width * dOmega) * (1./twopi)**D / energies[e], 1., rtol=0.001)

    continue

    for i in range(4, 8):
        km, fekm, _ = spectra_base.spectrum_integrate(kD, fekD,
            spec_type='modal', lenn=phys_dims,
            ignore_nan=False, norm_bin_size=False,
            bin_center=True, cut_excess=False,
            nan_small=False, num_bins=N//i)
        width = get_width(spectra_base.wavenumber_mesh(kD), N//i, False)
        if D == 2:
            dOmega = 2. * np.pi * km
        if D == 3:
            dOmega = 4. * np.pi * km**2
        print(np.nansum(fekm * width * dOmega) * (1./twopi)**D / energies[e])
        #plt.loglog(ko, feko, marker='.')
        #assert np.isclose(np.nansum(fekm * width * dOmega) * (1./twopi)**D / energies[e], 1., rtol=0.001)

    for i in range(4, 8):
        km, fekm, width = spectra_base.spectrum_integrate(kD, fekD,
            spec_type='modal', lenn=phys_dims,
            ignore_nan=False, log_space=True,
            norm_bin_size=False, bin_center=True,
            cut_excess=False, nan_small=False, num_bins=N//i)
        width = get_width(spectra_base.wavenumber_mesh(kD), N//i, True)
        if D == 2:
            dOmega = 2. * np.pi * km
        if D == 3:
            dOmega = 4. * np.pi * km**2
        print(np.nansum(fekm * width * dOmega) * (1./twopi)**D / energies[e])
        #plt.loglog(ko, feko, marker='.')
        #assert np.isclose(np.nansum(fekm * width * dOmega) * (1./twopi)**D / energies[e], 1., rtol=0.001)


1.2523022083587296
1.2075773980065203
1.1544780010734998
1.1089440097637524
1.0823788946044925
1.0694930145985433
1.0777129954552525
1.0827528222494143
1.0754667274460274
1.1126902775594567
1.1000501708760158
1.1242566055629009
1.1166881670345752
1.138995572575614
1.1435717579743772
1.146577571702697
1.1465826789984526
1.1510347814895636
1.1493918252665358
1.1487918480497192
